# Preprocessing — Customer Churn Prediction

**Project:** Customer Churn Prediction  
**Notebook:** 02 — Preprocessing  
**Referensi EDA:** Insight 1–69 (Fase 1–8B + Fase 9)  
**Keputusan:** `01_eda/01_eda_decisions_preprocessing.md`

---

## Alur Notebook

```
Raw Data
  └──→ Step 1 : Encode target variable (Churn → 0/1)              [Insight 5]
  └──→ Step 2 : Stratified split train/val/test                    [Insight 5]
  └──→ Step 3 : [FIT HANYA DI TRAINING SET MULAI DARI SINI]
  └──→ Step 4 : Feature engineering                                [Insight 39–44, 60–66]
  │             tc_residual, tenure_group (data-driven),
  │             monthly_to_total_ratio, is_auto_payment,
  │             service_count (kondisional), has_any_addon (kondisional)
  └──→ Step 5 : Drop kolom tidak berguna                           [Insight 1, 25, 45]
  │             id, gender, TotalCharges
  └──→ Step 6 : Structural encoder (No internet/phone → -1)        [Insight 6, 33, 34]
  └──→ Step 7 : Binary encoding (Yes/No → 1/0)                     [Insight 2, 15]
  └──→ Step 8 : One-Hot Encoding kolom nominal                      [Insight 12–14, 21]
  └──→ Step 9 : StandardScaler kolom numerik                        [Insight 7, 8, 10]
  └──→ Step 10: Save artifacts (preprocessor.joblib + splits)
```

---

**Perubahan dari versi sebelumnya (berdasarkan decisions doc):**
- `gender` sekarang di-DROP (Insight 25, 30 — selisih churn rate 0.57pp, chi² terkecil)
- `tc_residual` ditambahkan sebagai pengganti `TotalCharges` (Insight 64 — MI=0.032, range decile 29.12%)
- `tenure_group` menggunakan boundary **data-driven [0, 2, 18, 44, 72]** — 4 grup (Insight 66 — spread +12.47pp)
- `has_any_addon` ditambahkan versi kondisional (Insight 61 — spread 16.64pp)
- `service_count` dikonfirmasi kondisional — hitung hanya nilai `Yes` (Insight 60)

---
## Install & Import

In [1]:
!pip install imbalanced-learn --quiet
print('✅ Install selesai.')

✅ Install selesai.


In [2]:
# ── Standard Library ──────────────────────────────────────────────────────────
import os
import warnings
warnings.filterwarnings('ignore')

# ── Data Manipulation ─────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Sklearn — Preprocessing ───────────────────────────────────────────────────
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split

# ── Serialisasi ───────────────────────────────────────────────────────────────
import joblib

print('✅ Import selesai.')

✅ Import selesai.


---
## Konstanta Global

In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# KONSTANTA GLOBAL — Notebook Preprocessing
# ══════════════════════════════════════════════════════════════════════════════

# ── Path ──────────────────────────────────────────────────────────────────────
DATA_PATH         = '/kaggle/input/competitions/playground-series-s6e3/train.csv'
OUTPUT_DIR        = '/kaggle/working/artifacts'
PREPROCESSOR_PATH = f'{OUTPUT_DIR}/preprocessor.joblib'
SPLITS_DIR        = f'{OUTPUT_DIR}/splits'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(SPLITS_DIR,  exist_ok=True)

# ── Target & Identifier ───────────────────────────────────────────────────────
TARGET    = 'Churn'
CHURN_YES = 'Yes'
CHURN_NO  = 'No'
ID_COL    = 'id'

# ── Kolom yang di-drop ────────────────────────────────────────────────────────
# Keputusan dari decisions_preprocessing.md Langkah 1:
#   id         → identifier, tidak ada makna prediktif (Insight 1)
#   gender     → churn rate selisih 0.57pp, chi²=27.5 terkecil (Insight 25, 30)
#   TotalCharges → redundan 0.9921, digantikan tc_residual (Insight 45, 64)
DROP_COLS = ['id', 'gender', 'TotalCharges']

# ── Kolom numerik (sebelum feature engineering) ───────────────────────────────
NUMERIC_COLS = ['tenure', 'MonthlyCharges']

# ── Kolom add-on (untuk service_count & has_any_addon) ───────────────────────
ADDON_COLS = [
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies'
]

# ── Kolom yang punya structural dependency ────────────────────────────────────
# Kolom add-on: Yes=1, No=0, No internet service=-1 (Insight 33)
# MultipleLines: Yes=1, No=0, No phone service=-1   (Insight 34)
STRUCTURAL_COLS = ADDON_COLS + ['MultipleLines']

# ── Kolom binary Yes/No → 1/0 ────────────────────────────────────────────────
# gender sudah di-drop, SeniorCitizen sudah int (tidak disentuh)
BINARY_COLS = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']

# ── Kolom nominal untuk One-Hot Encoding ──────────────────────────────────────
# Contract → one-hot (bukan ordinal) — jarak antar kategori tidak setara (Insight 21)
OHE_COLS = ['Contract', 'InternetService', 'PaymentMethod']

# ── tenure_group — boundary data-driven dari Insight 66 ──────────────────────
# Domain   [0,12,36,72]: spread 42.43%, 3 grup
# Data-driven [0,2,18,44,72]: spread 54.90%, 4 grup → selisih +12.47pp
# Keputusan: DATA-DRIVEN (decisions_preprocessing.md Langkah 5B)
TENURE_BINS   = [0, 2, 18, 44, 72]
TENURE_LABELS = ['G1_0_2', 'G2_2_18', 'G3_18_44', 'G4_44_72']
# Churn rate per grup (Insight 66):
#   G1 (0–2 bln)  : 60.6%  ← sangat baru, paling berisiko
#   G2 (2–18 bln) : 41.3%  ← masa onboarding kritis
#   G3 (18–44 bln): 19.7%  ← transisi ke stabil
#   G4 (44–72 bln): 5.7%   ← loyal zone

# ── Payment method auto ───────────────────────────────────────────────────────
AUTO_PAYMENT_METHODS = [
    'Bank transfer (automatic)',
    'Credit card (automatic)'
]

# ── Split ratios ──────────────────────────────────────────────────────────────
TEST_SIZE   = 0.15
VAL_SIZE    = 0.15
RANDOM_SEED = 42

print('✅ Konstanta global terdefinisi.')
print(f'   DROP_COLS       : {DROP_COLS}')
print(f'   TENURE_BINS     : {TENURE_BINS}  (data-driven, Insight 66)')
print(f'   TENURE_LABELS   : {TENURE_LABELS}')
print(f'   RANDOM_SEED     : {RANDOM_SEED}')

✅ Konstanta global terdefinisi.
   DROP_COLS       : ['id', 'gender', 'TotalCharges']
   TENURE_BINS     : [0, 2, 18, 44, 72]  (data-driven, Insight 66)
   TENURE_LABELS   : ['G1_0_2', 'G2_2_18', 'G3_18_44', 'G4_44_72']
   RANDOM_SEED     : 42


---
## Deklarasi Class & Function

> Semua logic preprocessing dikemas dalam class sklearn-compatible
> (`BaseEstimator` + `TransformerMixin`) agar pipeline bisa
> di-serialize ke `preprocessor.joblib` dan di-load saat inference.
> Tidak ada eksekusi pada section ini — hanya definisi.

In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# CLASS: StructuralEncoder
# Tanggung jawab: handle nilai kondisional No internet/phone service
# Keputusan: decisions_preprocessing.md Langkah 4
# Dasar EDA: Insight 6, 33, 34
# ══════════════════════════════════════════════════════════════════════════════

class StructuralEncoder(BaseEstimator, TransformerMixin):
    """
    Encode 3 level semantik pada kolom add-on dan MultipleLines:
      'Yes'                 →  1  (aktif — punya akses dan ambil layanan)
      'No'                  →  0  (tidak aktif — punya akses tapi tidak ambil)
      'No internet service' → -1  (tidak relevan — tidak punya internet)
      'No phone service'    → -1  (tidak relevan — tidak punya telepon)

    Mengapa -1, bukan 0:
      Jika 'No' dan 'No internet service' keduanya → 0, model kehilangan
      informasi struktural bahwa pelanggan tanpa internet secara definitif
      tidak bisa punya add-on — bukan pilihan, tapi constraint.
      Tree-based model akan membuat split berbeda untuk -1 vs 0.

    Terkonfirmasi aman:
      0 pelanggaran structural dependency dari 140.727 baris (Insight 33)
      0 pelanggaran dari 36.301 baris PhoneService=No (Insight 34)
    """

    STRUCTURAL_MAP = {
        'Yes'                : 1,
        'No'                 : 0,
        'No internet service': -1,
        'No phone service'   : -1,
    }

    def __init__(self, cols: list = None):
        self.cols = cols

    def fit(self, X, y=None):
        self.cols_present_ = [c for c in (self.cols or []) if c in X.columns]
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.cols_present_:
            X[col] = X[col].map(self.STRUCTURAL_MAP).fillna(X[col])
        return X

    def get_feature_names_out(self, input_features=None):
        return input_features


print('✅ StructuralEncoder terdefinisi.')

✅ StructuralEncoder terdefinisi.


In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# CLASS: FeatureEngineer
# Tanggung jawab: membuat 6 fitur baru berdasarkan EDA Fase 5 + 8A + 8B
# Keputusan: decisions_preprocessing.md Langkah 5A–5F
# ══════════════════════════════════════════════════════════════════════════════

class FeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Membuat fitur-fitur baru berdasarkan bukti kuantitatif dari EDA.

    Urutan pembuatan KRITIS — beberapa fitur bergantung pada kolom asli
    yang akan di-drop atau di-encode di langkah berikutnya:

    1. tc_residual             (butuh TotalCharges SEBELUM di-drop)
       = TotalCharges - (tenure × MonthlyCharges)
       Menangkap sinyal non-linear residual yang tidak tertangkap oleh
       drop murni. MI=0.032, range decile churn rate 29.12% (Insight 64).

    2. monthly_to_total_ratio  (butuh TotalCharges SEBELUM di-drop)
       = MonthlyCharges / TotalCharges
       Proxy 'seberapa baru' pelanggan. Korelasi +0.3315 dengan Churn,
       lebih kuat dari MonthlyCharges mentah (Insight 41).

    3. tenure_group            (boundary data-driven [0,2,18,44,72])
       Spread 54.90% vs 42.43% domain boundary — selisih +12.47pp (Insight 66).
       4 grup: G1(0-2bln), G2(2-18bln), G3(18-44bln), G4(44-72bln).

    4. is_auto_payment         (butuh PaymentMethod SEBELUM di-OHE)
       Selisih churn rate 26.67%: non-auto 33.97% vs auto 7.30% (Insight 42).

    5. service_count           (KONDISIONAL — hitung hanya nilai 'Yes')
       Hanya hitung dari ADDON_COLS yang bernilai 'Yes'.
       'No internet service' tidak dihitung — constraint struktural, bukan pilihan.
       Tren monoton turun kondisional: 43.39%→42.27%→...→1.99% (Insight 60).

    6. has_any_addon           (KONDISIONAL — derived dari service_count)
       Versi biner dari service_count kondisional.
       Spread kondisional 16.64pp > threshold 10pp (Insight 61).

    TIDAK dibuat:
    - addon_to_monthly_ratio: korelasi -0.1088 terlalu lemah (Insight 44)
    """

    def __init__(
        self,
        tenure_bins   : list = None,
        tenure_labels : list = None,
        addon_cols    : list = None,
        auto_methods  : list = None
    ):
        self.tenure_bins   = tenure_bins   or TENURE_BINS
        self.tenure_labels = tenure_labels or TENURE_LABELS
        self.addon_cols    = addon_cols    or ADDON_COLS
        self.auto_methods  = auto_methods  or AUTO_PAYMENT_METHODS

    def fit(self, X, y=None):
        # Semua transformasi deterministik — tidak ada state yang perlu di-fit
        return self

    def transform(self, X):
        X = X.copy()

        # ── 1. tc_residual ────────────────────────────────────────────────────
        # Harus SEBELUM TotalCharges di-drop
        # Insight 64: MI residual=0.032, range decile 29.12% → sinyal non-linear valid
        if 'TotalCharges' in X.columns and 'tenure' in X.columns and 'MonthlyCharges' in X.columns:
            computed_total    = X['tenure'] * X['MonthlyCharges']
            X['tc_residual']  = X['TotalCharges'] - computed_total
        else:
            X['tc_residual']  = 0.0
            print('  ⚠ tc_residual tidak bisa dihitung — kolom yang dibutuhkan tidak ada')

        # ── 2. monthly_to_total_ratio ─────────────────────────────────────────
        # Harus SEBELUM TotalCharges di-drop
        # Insight 41: korelasi +0.3315 dengan Churn
        if 'TotalCharges' in X.columns and 'MonthlyCharges' in X.columns:
            total_safe                   = X['TotalCharges'].replace(0, np.nan)
            X['monthly_to_total_ratio']  = (X['MonthlyCharges'] / total_safe).fillna(1.0)

        # ── 3. tenure_group (data-driven boundary) ────────────────────────────
        # Insight 66: boundary [0,2,18,44,72] → spread 54.90% vs 42.43% domain
        if 'tenure' in X.columns:
            X['tenure_group'] = pd.cut(
                X['tenure'],
                bins   = self.tenure_bins,
                labels = self.tenure_labels,
                include_lowest = True
            ).astype(str)

        # ── 4. is_auto_payment ────────────────────────────────────────────────
        # Harus SEBELUM PaymentMethod di-OHE
        # Insight 42: selisih 26.67pp (non-auto 33.97% vs auto 7.30%)
        if 'PaymentMethod' in X.columns:
            X['is_auto_payment'] = X['PaymentMethod'].isin(
                self.auto_methods
            ).astype(int)

        # ── 5. service_count (kondisional — hanya hitung 'Yes') ───────────────
        # Insight 60: versi kondisional — 'No internet service' adalah constraint,
        # bukan pilihan → tidak dihitung sebagai 0 add-on
        addon_present = [c for c in self.addon_cols if c in X.columns]
        if addon_present:
            # Hitung hanya nilai 'Yes' — 'No internet service' → tidak terhitung
            X['service_count'] = X[addon_present].apply(
                lambda row: (row == 'Yes').sum(), axis=1
            ).astype(int)

        # ── 6. has_any_addon (kondisional — derived dari service_count) ───────
        # Insight 61: versi kondisional valid — spread 16.64pp > threshold 10pp
        if 'service_count' in X.columns:
            X['has_any_addon'] = (X['service_count'] > 0).astype(int)

        return X

    def get_feature_names_out(self, input_features=None):
        return input_features


print('✅ FeatureEngineer terdefinisi.')

✅ FeatureEngineer terdefinisi.


In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# CLASS: ColumnDropper
# Tanggung jawab: drop kolom yang tidak digunakan sebagai fitur
# Keputusan: decisions_preprocessing.md Langkah 1
# Dasar EDA: Insight 1 (id), Insight 25/30 (gender), Insight 45/64 (TotalCharges)
# ══════════════════════════════════════════════════════════════════════════════

class ColumnDropper(BaseEstimator, TransformerMixin):
    """
    Drop kolom yang sudah diputuskan tidak digunakan sebagai fitur.

    Kolom yang di-drop:
      id           → identifier (Insight 1)
      gender       → churn rate selisih 0.57pp, chi²=27.5 terkecil (Insight 25, 30)
      TotalCharges → redundan 0.9921, sudah digantikan tc_residual (Insight 45, 64)

    Drop dilakukan SETELAH feature engineering agar:
      - tc_residual dan monthly_to_total_ratio bisa memanfaatkan TotalCharges
      - Semua fitur baru sudah dibuat sebelum kolom asli dihapus
    """

    def __init__(self, cols_to_drop: list = None):
        self.cols_to_drop = cols_to_drop or DROP_COLS

    def fit(self, X, y=None):
        self.cols_dropped_ = [c for c in self.cols_to_drop if c in X.columns]
        self.cols_missing_  = [c for c in self.cols_to_drop if c not in X.columns]
        if self.cols_missing_:
            print(f'  ⚠ Kolom tidak ditemukan (mungkin sudah di-drop): {self.cols_missing_}')
        return self

    def transform(self, X):
        return X.drop(columns=self.cols_dropped_, errors='ignore')

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return None
        return [f for f in input_features if f not in self.cols_dropped_]


print('✅ ColumnDropper terdefinisi.')

✅ ColumnDropper terdefinisi.


In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# CLASS: BinaryEncoder
# Tanggung jawab: encode kolom binary Yes/No → 1/0
# Keputusan: decisions_preprocessing.md Langkah 6A
# Dasar EDA: Insight 2, 15
# ══════════════════════════════════════════════════════════════════════════════

class BinaryEncoder(BaseEstimator, TransformerMixin):
    """
    Encode kolom binary Yes/No menjadi 1/0.

    Kolom yang di-encode:
      Partner, Dependents, PhoneService, PaperlessBilling → Yes=1, No=0

    Tidak di-encode:
      gender      → sudah di-drop sebelum langkah ini
      SeniorCitizen → sudah int64 (0/1) dari source data (Insight 2)
                      tidak disentuh untuk menghindari double-encode

    Semua kolom binary dikonfirmasi hanya memiliki 2 nilai unik (Insight 6).
    Tidak ada whitespace atau casing inconsistency (Insight 6).
    """

    BINARY_MAP  = {'Yes': 1, 'No': 0}

    def __init__(self, cols: list = None):
        self.cols = cols or BINARY_COLS

    def fit(self, X, y=None):
        self.cols_present_ = [c for c in self.cols if c in X.columns]
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.cols_present_:
            X[col] = X[col].map(self.BINARY_MAP).fillna(X[col])
        return X

    def get_feature_names_out(self, input_features=None):
        return input_features


print('✅ BinaryEncoder terdefinisi.')

✅ BinaryEncoder terdefinisi.


In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# CLASS: OHEWrapper
# Tanggung jawab: One-Hot Encoding kolom nominal multi-kategori
# Keputusan: decisions_preprocessing.md Langkah 6B
# Dasar EDA: Insight 12–14, 21 (Contract jarak antar kategori tidak setara)
# ══════════════════════════════════════════════════════════════════════════════

class OHEWrapper(BaseEstimator, TransformerMixin):
    """
    One-Hot Encoding untuk kolom nominal.

    Kolom yang di-OHE:
      Contract       → 3 kategori, drop_first → 2 dummy
                       One-hot dipilih karena jarak antar kategori tidak setara:
                       M2M→One year: gap 36.29pp vs One year→Two year: gap 4.76pp (Insight 21)
      InternetService → 3 kategori → 2 dummy
      PaymentMethod   → 4 kategori → 3 dummy
      tenure_group    → 4 kategori (G1–G4) → 3 dummy (hasil feature engineering)

    Catatan: MultipleLines sudah di-handle oleh StructuralEncoder (-1/0/1)
    sehingga tidak perlu OHE. is_auto_payment juga sudah binary.

    Implementasi menggunakan sklearn OneHotEncoder dengan sparse_output=False
    agar output tetap DataFrame yang mudah diinspeksi.
    """

    def __init__(self, cols: list = None):
        self.cols = cols or (OHE_COLS + ['tenure_group'])
        self._encoder = OneHotEncoder(
            drop          = 'first',
            sparse_output = False,
            handle_unknown= 'ignore',
            dtype         = np.float64
        )

    def fit(self, X, y=None):
        self.cols_present_ = [c for c in self.cols if c in X.columns]
        if self.cols_present_:
            self._encoder.fit(X[self.cols_present_])
            self.ohe_feature_names_ = self._encoder.get_feature_names_out(
                self.cols_present_
            ).tolist()
        else:
            self.ohe_feature_names_ = []
        return self

    def transform(self, X):
        X = X.copy()
        if not self.cols_present_:
            return X

        ohe_array = self._encoder.transform(X[self.cols_present_])
        ohe_df    = pd.DataFrame(
            ohe_array,
            columns = self.ohe_feature_names_,
            index   = X.index
        )
        X = X.drop(columns=self.cols_present_)
        X = pd.concat([X, ohe_df], axis=1)
        return X

    def get_feature_names_out(self, input_features=None):
        return input_features


print('✅ OHEWrapper terdefinisi.')

✅ OHEWrapper terdefinisi.


In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# CLASS: ScalerWrapper
# Tanggung jawab: StandardScaler pada kolom numerik kontinu
# Keputusan: decisions_preprocessing.md Langkah 7
# Dasar EDA: Insight 7 (skewness tenure 0.063), Insight 8 (skewness monthly -0.29),
#            Insight 10 (0 outlier — StandardScaler aman tanpa robust variant)
# ══════════════════════════════════════════════════════════════════════════════

class ScalerWrapper(BaseEstimator, TransformerMixin):
    """
    StandardScaler (z-score) untuk fitur numerik kontinu.

    Kolom yang di-scale:
      tenure                → skewness 0.063, mendekati simetris (Insight 7)
      MonthlyCharges        → skewness -0.29, masih dalam batas aman (Insight 8)
      tc_residual           → fitur baru, distribusi kontinu
      monthly_to_total_ratio → fitur baru, distribusi kontinu

    Tidak di-scale (walaupun numerik):
      SeniorCitizen     → binary 0/1, scaling tidak bermakna
      is_auto_payment   → binary 0/1
      service_count     → integer diskret 0–6
      has_any_addon     → binary 0/1
      Semua hasil OHE   → binary 0/1

    StandardScaler dipilih (bukan RobustScaler) karena:
      0 outlier di semua kolom numerik berdasarkan IQR (Insight 10).
      Jika ada outlier, RobustScaler lebih tepat — tapi di dataset ini tidak ada.

    Scaling diterapkan meskipun model utama adalah tree-based karena:
      Pipeline harus support eksperimen dengan Logistic Regression (Tier 2)
      yang memerlukan scaling. Scaling tidak merugikan tree-based model.
    """

    # Kolom yang di-scale — dikonfirmasi di fit() agar robust terhadap
    # perubahan nama kolom hasil feature engineering
    NUMERIC_TARGET_COLS = [
        'tenure', 'MonthlyCharges',
        'tc_residual', 'monthly_to_total_ratio'
    ]

    def __init__(self, cols: list = None):
        self.cols    = cols or self.NUMERIC_TARGET_COLS
        self._scaler = StandardScaler()

    def fit(self, X, y=None):
        self.cols_present_ = [c for c in self.cols if c in X.columns]
        if self.cols_present_:
            self._scaler.fit(X[self.cols_present_])
        return self

    def transform(self, X):
        X = X.copy()
        if self.cols_present_:
            X[self.cols_present_] = self._scaler.transform(X[self.cols_present_])
        return X

    def get_feature_names_out(self, input_features=None):
        return input_features


print('✅ ScalerWrapper terdefinisi.')

✅ ScalerWrapper terdefinisi.


In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# CLASS: PreprocessingPipeline
# Tanggung jawab: orkestrasi seluruh step dalam urutan yang benar
# Output: DataFrame numerik siap untuk training
# ══════════════════════════════════════════════════════════════════════════════

class PreprocessingPipeline(BaseEstimator, TransformerMixin):
    """
    Pipeline preprocessing end-to-end.

    Urutan step (KRITIS — tidak boleh diubah):
      1. FeatureEngineer
         tc_residual dan monthly_to_total_ratio HARUS dihitung sebelum
         TotalCharges di-drop di step berikutnya.

      2. ColumnDropper
         Drop id, gender, TotalCharges SETELAH feature engineering selesai.

      3. StructuralEncoder
         Encode No internet/phone service → -1 SEBELUM BinaryEncoder
         agar nilai 'No' biasa tidak tertimpa.

      4. BinaryEncoder
         Encode Yes/No → 1/0 untuk kolom binary.

      5. OHEWrapper
         One-Hot Encoding untuk Contract, InternetService, PaymentMethod,
         dan tenure_group (yang dibuat di step 1).

      6. ScalerWrapper
         StandardScaler untuk tenure, MonthlyCharges, tc_residual,
         monthly_to_total_ratio.

    Prinsip:
      - fit() hanya boleh dipanggil pada training data (no leakage)
      - transform() dipanggil pada val dan test menggunakan parameter dari fit()
      - Semua step adalah sklearn-compatible → serializable via joblib
    """

    def __init__(self):
        self.feature_engineer_   = FeatureEngineer()
        self.col_dropper_        = ColumnDropper()
        self.structural_encoder_ = StructuralEncoder(cols=STRUCTURAL_COLS)
        self.binary_encoder_     = BinaryEncoder()
        self.ohe_wrapper_        = OHEWrapper()
        self.scaler_wrapper_     = ScalerWrapper()

        self._steps = [
            ('feature_engineer',   self.feature_engineer_),
            ('col_dropper',        self.col_dropper_),
            ('structural_encoder', self.structural_encoder_),
            ('binary_encoder',     self.binary_encoder_),
            ('ohe_wrapper',        self.ohe_wrapper_),
            ('scaler_wrapper',     self.scaler_wrapper_),
        ]

    def fit(self, X, y=None):
        X_transformed = X.copy()
        for name, step in self._steps:
            X_transformed = step.fit_transform(X_transformed, y)
        return self

    def transform(self, X):
        X_transformed = X.copy()
        for name, step in self._steps:
            X_transformed = step.transform(X_transformed)
        return X_transformed

    def fit_transform(self, X, y=None):
        X_transformed = X.copy()
        for name, step in self._steps:
            X_transformed = step.fit_transform(X_transformed, y)
        return X_transformed

    def get_feature_names(self):
        """Kembalikan nama fitur output setelah seluruh pipeline."""
        return list(self._last_output_columns_) if hasattr(self, '_last_output_columns_') else []

    def fit_transform(self, X, y=None):
        X_transformed = X.copy()
        for name, step in self._steps:
            X_transformed = step.fit_transform(X_transformed, y)
        self._last_output_columns_ = X_transformed.columns.tolist()
        return X_transformed

    def transform(self, X):
        X_transformed = X.copy()
        for name, step in self._steps:
            X_transformed = step.transform(X_transformed)
        return X_transformed


print('✅ PreprocessingPipeline terdefinisi.')

✅ PreprocessingPipeline terdefinisi.


In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# CLASS: DataSplitter
# Tanggung jawab: stratified split train/val/test
# Keputusan: decisions_preprocessing.md Langkah 3
# Dasar EDA: Insight 5 (imbalance 3.44x — stratified wajib)
# ══════════════════════════════════════════════════════════════════════════════

class DataSplitter:
    """
    Stratified train/val/test split.

    Stratified diperlukan karena imbalance 3.44x (Insight 5):
    tanpa stratifikasi, val/test bisa berisi proporsi minority class
    yang berbeda jauh dari distribusi aslinya → evaluasi tidak representatif.

    Split dilakukan SEBELUM fit preprocessor untuk mencegah data leakage:
    parameter preprocessor (mean, std, OHE categories) tidak boleh
    dipengaruhi oleh data val/test.
    """

    def __init__(
        self,
        test_size    : float = TEST_SIZE,
        val_size     : float = VAL_SIZE,
        random_state : int   = RANDOM_SEED
    ):
        self.test_size    = test_size
        self.val_size     = val_size
        self.random_state = random_state

    def split(self, X, y):
        # Step 1: pisahkan test set dari keseluruhan data
        X_trainval, X_test, y_trainval, y_test = train_test_split(
            X, y,
            test_size    = self.test_size,
            stratify     = y,
            random_state = self.random_state
        )

        # Step 2: pisahkan val set dari trainval
        # val_size relatif terhadap TOTAL — perlu penyesuaian
        val_size_adjusted = self.val_size / (1 - self.test_size)
        X_train, X_val, y_train, y_val = train_test_split(
            X_trainval, y_trainval,
            test_size    = val_size_adjusted,
            stratify     = y_trainval,
            random_state = self.random_state
        )

        return X_train, X_val, X_test, y_train, y_val, y_test

    @staticmethod
    def report_split(y_train, y_val, y_test):
        for name, y in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
            n       = len(y)
            n_churn = y.sum()
            pct     = n_churn / n * 100
            print(f'  {name:5s}: n={n:>7,} | Churn=1: {n_churn:>6,} ({pct:.1f}%)')


print('✅ DataSplitter terdefinisi.')
print()
print('✅ Semua class terdefinisi. Lanjut ke eksekusi.')

✅ DataSplitter terdefinisi.

✅ Semua class terdefinisi. Lanjut ke eksekusi.


---
## Eksekusi Preprocessing

### Step 1 — Load Data

In [12]:
print('Loading data...')
df_raw = pd.read_csv(DATA_PATH)
print(f'Shape    : {df_raw.shape[0]:,} baris × {df_raw.shape[1]} kolom')
print(f'Kolom    : {df_raw.columns.tolist()}')
print(f'Missing  : {df_raw.isnull().sum().sum()} total')
df_raw.head(3)

Loading data...
Shape    : 594,194 baris × 21 kolom
Kolom    : ['id', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']
Missing  : 0 total


,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.1,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.5,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.4,5841.35,No


### Step 2 — Encode Target & Split

In [13]:
# ── Encode target: Churn Yes→1, No→0 (Insight 5) ────────────────────────────
X_raw = df_raw.drop(columns=[TARGET])
y_raw = (df_raw[TARGET] == CHURN_YES).astype(int)

print('Distribusi target (Insight 5):')
vc = y_raw.value_counts().rename({0: 'No Churn (0)', 1: 'Churn (1)'})
print(vc.to_string())
print(f'\nImbalance ratio: {(y_raw==0).sum()/(y_raw==1).sum():.2f}x')

Distribusi target (Insight 5):
Churn
No Churn (0)    460377
Churn (1)       133817

Imbalance ratio: 3.44x


In [14]:
# ── Stratified split (SEBELUM fit preprocessor — no leakage) ─────────────────
print('\nStratified split (test=15%, val=15% dari total)...')
splitter = DataSplitter(
    test_size    = TEST_SIZE,
    val_size     = VAL_SIZE,
    random_state = RANDOM_SEED
)
X_train_raw, X_val_raw, X_test_raw, y_train, y_val, y_test = splitter.split(X_raw, y_raw)

print('\nRingkasan split:')
DataSplitter.report_split(y_train, y_val, y_test)


Stratified split (test=15%, val=15% dari total)...

Ringkasan split:
  Train: n=415,935 | Churn=1: 93,672 (22.5%)
  Val  : n= 89,129 | Churn=1: 20,072 (22.5%)
  Test : n= 89,130 | Churn=1: 20,073 (22.5%)


### Step 3–9 — Fit Preprocessor & Transform

> Preprocessor di-**fit hanya pada training data**.
> Val dan test di-**transform** menggunakan parameter yang sudah di-fit.

In [15]:
# ── Fit pada training set ─────────────────────────────────────────────────────
print('Fitting preprocessing pipeline pada X_train...')
preprocessor  = PreprocessingPipeline()
X_train_proc  = preprocessor.fit_transform(X_train_raw)

print(f'\nShape setelah preprocessing:')
print(f'  X_train_raw  : {X_train_raw.shape}')
print(f'  X_train_proc : {X_train_proc.shape}')

# Fitur baru
new_cols     = [c for c in X_train_proc.columns if c not in X_train_raw.columns]
print(f'\nFitur baru ditambahkan ({len(new_cols)}): {new_cols}')

# Kolom yang di-drop
dropped_cols = [c for c in X_train_raw.columns if c not in X_train_proc.columns and c not in new_cols]
print(f'Kolom yang di-drop       : {dropped_cols}')

Fitting preprocessing pipeline pada X_train...

Shape setelah preprocessing:
  X_train_raw  : (415935, 20)
  X_train_proc : (415935, 29)

Fitur baru ditambahkan (15): ['tc_residual', 'monthly_to_total_ratio', 'is_auto_payment', 'service_count', 'has_any_addon', 'Contract_One year', 'Contract_Two year', 'InternetService_Fiber optic', 'InternetService_No', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check', 'tenure_group_G2_2_18', 'tenure_group_G3_18_44', 'tenure_group_G4_44_72']
Kolom yang di-drop       : ['id', 'gender', 'InternetService', 'Contract', 'PaymentMethod', 'TotalCharges']


In [16]:
# ── Transform val dan test ────────────────────────────────────────────────────
X_val_proc  = preprocessor.transform(X_val_raw)
X_test_proc = preprocessor.transform(X_test_raw)

print('Transform val dan test selesai.')
print(f'  X_val_proc  : {X_val_proc.shape}')
print(f'  X_test_proc : {X_test_proc.shape}')

# Konfirmasi tidak ada missing value
print()
for name, X in [('train', X_train_proc), ('val', X_val_proc), ('test', X_test_proc)]:
    n_miss  = X.isnull().sum().sum()
    status  = '✅ 0 missing' if n_miss == 0 else f'⚠ {n_miss} missing'
    print(f'  Missing values {name:5s}: {status}')

Transform val dan test selesai.
  X_val_proc  : (89129, 29)
  X_test_proc : (89130, 29)

  Missing values train: ✅ 0 missing
  Missing values val  : ✅ 0 missing
  Missing values test : ✅ 0 missing


---
### Validasi Preprocessing

In [17]:
print('=== VALIDASI PREPROCESSING ===')
print()

errors = []

# 1. Kolom yang seharusnya di-drop
for col in DROP_COLS:
    if col in X_train_proc.columns:
        errors.append(f'❌ {col} seharusnya sudah di-drop')
    else:
        print(f'✅ {col} sudah di-drop')

# 2. Fitur baru yang seharusnya ada
required_new = [
    'tc_residual', 'monthly_to_total_ratio',
    'tenure_group_G2_2_18', 'tenure_group_G3_18_44', 'tenure_group_G4_44_72',
    'is_auto_payment', 'service_count', 'has_any_addon'
]
for col in required_new:
    if col not in X_train_proc.columns:
        errors.append(f'❌ {col} tidak ditemukan — cek FeatureEngineer')
    else:
        print(f'✅ {col} ada')

# 3. tenure_group: pastikan 4 grup (data-driven) — bukan 3 grup (domain)
tg_cols = [c for c in X_train_proc.columns if 'tenure_group' in c]
if len(tg_cols) == 3:
    print(f'✅ tenure_group: 4 grup (data-driven) → {len(tg_cols)} dummy kolom: {tg_cols}')
else:
    errors.append(f'❌ tenure_group: ekspektasi 3 dummy (4 grup), ditemukan {len(tg_cols)}: {tg_cols}')

# 4. SeniorCitizen masih ada
if 'SeniorCitizen' not in X_train_proc.columns:
    errors.append('❌ SeniorCitizen tidak ada — seharusnya dipertahankan')
else:
    print(f'✅ SeniorCitizen ada (sudah 0/1 dari source, Insight 2)')

# 5. tenure scaled (mean ≈ 0)
tenure_mean = X_train_proc['tenure'].mean()
if abs(tenure_mean) < 0.01:
    print(f'✅ tenure scaled: mean={tenure_mean:.4f} ≈ 0 (StandardScaler, Insight 7)')
else:
    errors.append(f'❌ tenure tidak di-scale: mean={tenure_mean:.4f}')

# 6. tc_residual tersedia dan bukan semua nol
if 'tc_residual' in X_train_proc.columns:
    n_zeros = (X_train_proc['tc_residual'] == 0).sum()
    if n_zeros < len(X_train_proc) * 0.5:
        print(f'✅ tc_residual berisi nilai valid (Insight 64)')
    else:
        errors.append(f'❌ tc_residual terlalu banyak nol — cek implementasi')

# 7. OHE Contract → drop first → 2 dummy
contract_cols = [c for c in X_train_proc.columns if 'Contract' in c]
if len(contract_cols) == 2:
    print(f'✅ Contract OHE: {contract_cols}')
else:
    errors.append(f'❌ Contract OHE: ekspektasi 2 kolom, ditemukan {len(contract_cols)}: {contract_cols}')

print()
if errors:
    print(f'⚠ {len(errors)} validasi GAGAL:')
    for e in errors:
        print(f'  {e}')
else:
    print('✅ Semua validasi preprocessing LULUS.')

=== VALIDASI PREPROCESSING ===

✅ id sudah di-drop
✅ gender sudah di-drop
✅ TotalCharges sudah di-drop
✅ tc_residual ada
✅ monthly_to_total_ratio ada
✅ tenure_group_G2_2_18 ada
✅ tenure_group_G3_18_44 ada
✅ tenure_group_G4_44_72 ada
✅ is_auto_payment ada
✅ service_count ada
✅ has_any_addon ada
✅ tenure_group: 4 grup (data-driven) → 3 dummy kolom: ['tenure_group_G2_2_18', 'tenure_group_G3_18_44', 'tenure_group_G4_44_72']
✅ SeniorCitizen ada (sudah 0/1 dari source, Insight 2)
✅ tenure scaled: mean=0.0000 ≈ 0 (StandardScaler, Insight 7)
✅ tc_residual berisi nilai valid (Insight 64)
✅ Contract OHE: ['Contract_One year', 'Contract_Two year']

✅ Semua validasi preprocessing LULUS.


---
### Ringkasan Fitur Output

In [18]:
print('=== RINGKASAN FITUR OUTPUT PREPROCESSING ===')
print(f'\nTotal fitur: {X_train_proc.shape[1]}')
print()

# Kelompokkan fitur berdasarkan jenisnya
fitur_numerik    = ['tenure', 'MonthlyCharges', 'tc_residual', 'monthly_to_total_ratio']
fitur_binary     = ['SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
                    'PaperlessBilling', 'is_auto_payment', 'has_any_addon']
fitur_diskret    = ['service_count']
fitur_structural = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                    'TechSupport', 'StreamingTV', 'StreamingMovies', 'MultipleLines']
fitur_ohe        = [c for c in X_train_proc.columns
                    if any(c.startswith(p) for p in
                           ['Contract_', 'InternetService_', 'PaymentMethod_', 'tenure_group_'])]

print('Numerik kontinu (di-scale):')
for c in fitur_numerik:
    status = '✅' if c in X_train_proc.columns else '❌ TIDAK ADA'
    print(f'  {status} {c}')

print('\nBinary 0/1:')
for c in fitur_binary:
    status = '✅' if c in X_train_proc.columns else '❌ TIDAK ADA'
    print(f'  {status} {c}')

print('\nInteger diskret:')
for c in fitur_diskret:
    status = '✅' if c in X_train_proc.columns else '❌ TIDAK ADA'
    print(f'  {status} {c}')

print('\nStructural (-1/0/1):')
for c in fitur_structural:
    status = '✅' if c in X_train_proc.columns else '❌ TIDAK ADA'
    print(f'  {status} {c}')

print(f'\nOne-Hot Encoding ({len(fitur_ohe)} kolom):')
for c in sorted(fitur_ohe):
    print(f'  ✅ {c}')

all_accounted = set(fitur_numerik + fitur_binary + fitur_diskret +
                    fitur_structural + fitur_ohe)
leftover      = [c for c in X_train_proc.columns if c not in all_accounted]
if leftover:
    print(f'\nKolom lain: {leftover}')

=== RINGKASAN FITUR OUTPUT PREPROCESSING ===

Total fitur: 29

Numerik kontinu (di-scale):
  ✅ tenure
  ✅ MonthlyCharges
  ✅ tc_residual
  ✅ monthly_to_total_ratio

Binary 0/1:
  ✅ SeniorCitizen
  ✅ Partner
  ✅ Dependents
  ✅ PhoneService
  ✅ PaperlessBilling
  ✅ is_auto_payment
  ✅ has_any_addon

Integer diskret:
  ✅ service_count

Structural (-1/0/1):
  ✅ OnlineSecurity
  ✅ OnlineBackup
  ✅ DeviceProtection
  ✅ TechSupport
  ✅ StreamingTV
  ✅ StreamingMovies
  ✅ MultipleLines

One-Hot Encoding (10 kolom):
  ✅ Contract_One year
  ✅ Contract_Two year
  ✅ InternetService_Fiber optic
  ✅ InternetService_No
  ✅ PaymentMethod_Credit card (automatic)
  ✅ PaymentMethod_Electronic check
  ✅ PaymentMethod_Mailed check
  ✅ tenure_group_G2_2_18
  ✅ tenure_group_G3_18_44
  ✅ tenure_group_G4_44_72


---
### Save Artifacts

In [19]:
# ── Save preprocessor.joblib ──────────────────────────────────────────────────
joblib.dump(preprocessor, PREPROCESSOR_PATH)
print(f'✅ Preprocessor disimpan: {PREPROCESSOR_PATH}')
print(f'   Total fitur output   : {X_train_proc.shape[1]}')

✅ Preprocessor disimpan: /kaggle/working/artifacts/preprocessor.joblib
   Total fitur output   : 29


In [20]:
# ── Save splits sebagai CSV ───────────────────────────────────────────────────
splits_csv = {
    'X_train': X_train_proc, 'X_val': X_val_proc, 'X_test': X_test_proc,
    'y_train': y_train,      'y_val': y_val,       'y_test': y_test
}
for name, data in splits_csv.items():
    path = f'{SPLITS_DIR}/{name}.csv'
    data.to_csv(path, index=False)
    kb   = os.path.getsize(path) / 1024
    print(f'  ✅ {name}.csv  ({kb:.0f} KB)')

# ── Save splits sebagai joblib (lebih cepat di-load) ─────────────────────────
splits_joblib_path = f'{OUTPUT_DIR}/splits.joblib'
joblib.dump({
    'X_train'        : X_train_proc.values,
    'X_val'          : X_val_proc.values,
    'X_test'         : X_test_proc.values,
    'y_train'        : y_train.values,
    'y_val'          : y_val.values,
    'y_test'         : y_test.values,
    'feature_names'  : X_train_proc.columns.tolist(),
    'imbalance_ratio': (y_raw==0).sum() / (y_raw==1).sum(),
    'random_seed'    : RANDOM_SEED
}, splits_joblib_path)
print(f'\n✅ Splits joblib disimpan: {splits_joblib_path}')
print(f'   Keys: {list(joblib.load(splits_joblib_path).keys())}')

  ✅ X_train.csv  (61133 KB)
  ✅ X_val.csv  (13102 KB)
  ✅ X_test.csv  (13101 KB)
  ✅ y_train.csv  (812 KB)
  ✅ y_val.csv  (174 KB)
  ✅ y_test.csv  (174 KB)

✅ Splits joblib disimpan: /kaggle/working/artifacts/splits.joblib
   Keys: ['X_train', 'X_val', 'X_test', 'y_train', 'y_val', 'y_test', 'feature_names', 'imbalance_ratio', 'random_seed']


---
## Ringkasan Final

In [21]:
print('═' * 60)
print('  PREPROCESSING SELESAI')
print('═' * 60)
print()
print('Artifacts yang dihasilkan:')
print(f'  {PREPROCESSOR_PATH}')
print(f'  {splits_joblib_path}')
print()
print('Perubahan dari versi sebelumnya:')
print('  ✅ gender di-DROP  (chi²=27.5 terkecil, Insight 25, 30)')
print('  ✅ tc_residual DIBUAT sebagai pengganti TotalCharges (Insight 64)')
print('  ✅ tenure_group boundary DATA-DRIVEN [0,2,18,44,72] (Insight 66)')
print('  ✅ has_any_addon DITAMBAHKAN versi kondisional (Insight 61)')
print()
print('Load di notebook berikutnya:')
print('  import joblib')
print('  splits = joblib.load("../artifacts/splits.joblib")')
print('  X_train = splits["X_train"]')
print('  feature_names = splits["feature_names"]')
print()
print('Langkah berikutnya: 03_baseline/03_baseline.ipynb')

════════════════════════════════════════════════════════════
  PREPROCESSING SELESAI
════════════════════════════════════════════════════════════

Artifacts yang dihasilkan:
  /kaggle/working/artifacts/preprocessor.joblib
  /kaggle/working/artifacts/splits.joblib

Perubahan dari versi sebelumnya:
  ✅ gender di-DROP  (chi²=27.5 terkecil, Insight 25, 30)
  ✅ tc_residual DIBUAT sebagai pengganti TotalCharges (Insight 64)
  ✅ tenure_group boundary DATA-DRIVEN [0,2,18,44,72] (Insight 66)
  ✅ has_any_addon DITAMBAHKAN versi kondisional (Insight 61)

Load di notebook berikutnya:
  import joblib
  splits = joblib.load("../artifacts/splits.joblib")
  X_train = splits["X_train"]
  feature_names = splits["feature_names"]

Langkah berikutnya: 03_baseline/03_baseline.ipynb
